# QUDORA trapped-ion emulators: working with the noise model

> **Billing notice:** both QUDORA simulators are currently **free to run** (0 credits). You need a qBraid API key, but no QUDORA account.

[QUDORA](https://www.qudora.com) builds trapped-ion quantum computers controlled by microwaves
instead of lasers. qBraid offers two of its simulators:

| Device | Device ID | Qubits |
|---|---|---|
| Qamelion | `qudora:qudora:sim:qamelion` | 32 |
| QVLS-Q1 Emulator | `qudora:qudora:sim:qvls-q1-emulator` | 20 |

Both are state vector simulators that apply **QUDORA's trapped-ion noise model by default**, so
their counts look like what QUDORA's hardware would return. Four settings let you tune that model
or switch it off, and that is what this notebook is about:

1. A first run, with and without noise
2. What each noise setting means
3. Sweeping the measurement error, and checking it against theory
4. Sweeping the dephasing time T2
5. Many circuits in one batch job

In [ ]:
%%capture

# "%%capture" hides the install output; comment it out if you need to debug this step.
%pip install 'qbraid[qiskit,visualization]'

## Connect to the devices

`QbraidProvider` picks up your API key from `~/.qbraid/qbraidrc` or the `QBRAID_API_KEY`
environment variable. Inside qBraid Lab it is already configured for you.

In [ ]:
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()
qamelion = provider.get_device("qudora:qudora:sim:qamelion")
qvls = provider.get_device("qudora:qudora:sim:qvls-q1-emulator")

for device in (qamelion, qvls):
    meta = device.metadata()
    print(meta["name"], "|", meta["num_qubits"], "qubits |", meta["status"])

Both take OpenQASM 2 or 3, and qBraid transpiles to it for you, so a Qiskit, Cirq, Braket or
pytket circuit can go straight to `run()`. QUDORA then compiles it to its native trapped-ion
gates, `r` (single-qubit rotations) and `rxx` (the two-qubit Mølmer–Sørensen interaction).

## A first run, with and without noise

We use GHZ states throughout: ideally, every qubit agrees, so all shots land on all-zeros or
all-ones. Any other outcome is the noise model at work.

In [ ]:
from qiskit import QuantumCircuit


def ghz(num_qubits: int) -> QuantumCircuit:
    circuit = QuantumCircuit(num_qubits)
    circuit.h(0)
    for qubit in range(num_qubits - 1):
        circuit.cx(qubit, qubit + 1)
    circuit.measure_all()
    return circuit


def ghz_population(counts: dict, num_qubits: int) -> float:
    """Fraction of shots on all-zeros or all-ones."""
    shots = sum(counts.values())
    return (counts.get("0" * num_qubits, 0) + counts.get("1" * num_qubits, 0)) / shots


circuit = ghz(3)
circuit.draw()

Noise settings go in `runtime_options`, under **`backend_settings`**. To switch the model off,
set the three error settings to `0.0` and the dephasing time to `None`:

In [ ]:
NOISELESS = {
    "measurement_error_probability": 0.0,
    "single_qubit_gate_noise_strength": 0.0,
    "two_qubit_gate_noise_strength": 0.0,
    "dephasing_T2_time": None,
}

noisy_job = qamelion.run(circuit, shots=2000)
ideal_job = qamelion.run(circuit, shots=2000, runtime_options={"backend_settings": NOISELESS})

noisy_counts = noisy_job.result().data.get_counts()
ideal_counts = ideal_job.result().data.get_counts()

print("default noise:", noisy_counts)
print("noise off:    ", ideal_counts)

In [ ]:
from qbraid.visualization import plot_distribution

plot_distribution(
    [noisy_counts, ideal_counts],
    legend=["QUDORA noise model (default)", "Noise off"],
    title="3-qubit GHZ state on Qamelion",
)

With noise off, only `000` and `111` appear. With the default model, a small fraction of shots
lands elsewhere.

## The four noise settings

| Setting | Type | Default | Range | What it means |
|---|---|---|---|---|
| `measurement_error_probability` | float | `0.0035` | 0 to 1 | The probability that a qubit's readout is wrong |
| `single_qubit_gate_noise_strength` | float | `1.0` | 0 to 1 | A **scale factor** on QUDORA's single-qubit gate noise: `1.0` is the full calibrated model, `0.0` is off |
| `two_qubit_gate_noise_strength` | float | `1.0` | 0 to 1 | The same scale factor, for two-qubit gates |
| `dephasing_T2_time` | float or `None` | `60` | above 0 | Dephasing time T2, in **seconds**. Shorter is noisier; `None` switches dephasing off |

Note the difference between the first setting and the two gate settings: the measurement error
is a probability you set directly, while the gate settings scale QUDORA's own gate noise model up
or down. A gate noise strength of `0.5` means half of QUDORA's modelled gate noise, not a 50%
error rate.

You don't have to pass all four. Any setting you leave out keeps its default, so this changes
only the readout error:

```python
qamelion.run(circuit, shots=2000, runtime_options={"backend_settings": {"measurement_error_probability": 0.0}})
```

Settings apply to a whole job. To compare settings, submit one job per setting; `run()` returns
straight away, so the jobs run side by side.

## Sweeping the measurement error

To see one setting on its own, switch everything else off and vary the readout error `p`. On an
`n`-qubit GHZ state, a shot stays on all-zeros or all-ones only if no qubit is misread, or (much
less likely) every qubit is, so the expected GHZ population is $(1-p)^n + p^n$:

In [ ]:
import numpy as np

n = 5
error_rates = [0.0, 0.02, 0.05, 0.1, 0.2, 0.3]

readout_jobs = [
    qvls.run(
        ghz(n),
        shots=2000,
        runtime_options={"backend_settings": {**NOISELESS, "measurement_error_probability": p}},
    )
    for p in error_rates
]
readout_population = [ghz_population(job.result().data.get_counts(), n) for job in readout_jobs]

for p, population in zip(error_rates, readout_population):
    print(f"p = {p:.2f}: GHZ population {population:.3f}, expected {(1 - p) ** n + p ** n:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fine = np.linspace(0, 0.3, 100)
plt.plot(fine, (1 - fine) ** n + fine**n, label=r"$(1-p)^5 + p^5$")
plt.plot(error_rates, readout_population, "o", label="QVLS-Q1 Emulator")
plt.xlabel("measurement_error_probability")
plt.ylabel("GHZ population")
plt.title("5-qubit GHZ state, readout error only")
plt.legend()
plt.show()

## Sweeping the dephasing time T2

Dephasing scrambles the relative phase between the GHZ state's two halves while the circuit runs.
The default T2 of 60 seconds is long compared with QUDORA's gate times, so it hardly shows. Shrink
it, with every other source of noise off, and the GHZ population falls away once T2 approaches
the circuit's duration:

In [ ]:
t2_times = [10, 1, 0.1, 0.03, 0.01, 0.003, 0.001]

t2_jobs = [
    qvls.run(
        ghz(n),
        shots=2000,
        runtime_options={"backend_settings": {**NOISELESS, "dephasing_T2_time": t2}},
    )
    for t2 in t2_times
]
t2_population = [ghz_population(job.result().data.get_counts(), n) for job in t2_jobs]

plt.semilogx(t2_times, t2_population, "o-")
plt.xlabel("dephasing_T2_time (seconds)")
plt.ylabel("GHZ population")
plt.title("5-qubit GHZ state, dephasing only")
plt.show()

## Many circuits in one batch job

Both devices support **batch jobs**: pass a list of circuits with `as_batch=True` and they run as
**one job**, with one set of counts per circuit, in order. Every circuit in a batch shares the
job's settings, which suits a sweep over circuits rather than over settings.

Here we grow the GHZ state from 2 to 10 qubits under QUDORA's default noise, all in one job, to
see how the noise accumulates as the circuit gets bigger. Simulating noise costs more per shot
than an ideal run, and grows quickly with width, so this keeps to 10 qubits and 1,000 shots:

In [ ]:
widths = list(range(2, 11, 2))

batch_job = qamelion.run([ghz(width) for width in widths], shots=1000, as_batch=True)
batch_counts = batch_job.result().data.get_counts()  # one dict per circuit

width_population = [ghz_population(counts, width) for counts, width in zip(batch_counts, widths)]

plt.plot(widths, width_population, "o-")
plt.xlabel("GHZ width (qubits)")
plt.ylabel("GHZ population")
plt.title("QUDORA default noise, one batch job on Qamelion")
plt.show()

A QUDORA batch takes up to **500 circuits**, each run with the same number of shots, up to
**10,000**. Without `as_batch=True`, a list of circuits is submitted as separate jobs, one per
circuit.

## Summary

- Qamelion (32 qubits) and the QVLS-Q1 Emulator (20 qubits) are free on `QbraidProvider`, and apply QUDORA's trapped-ion noise model by default.
- Noise settings go in `runtime_options={"backend_settings": {...}}`. Any you leave out keep their defaults.
- `measurement_error_probability` is a probability; the two gate noise strengths are scale factors on QUDORA's own model; `dephasing_T2_time` is in seconds, and `None` switches it off.
- Settings apply per job, so sweep a setting with one job per value. Sweep circuits with one batch job.
- More detail on the model: [QUDORA's noise model documentation](https://cloud.qudora.com/main/documentation#appendix--noise-model-details).

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>